# 02 — Modelado Predictivo

**Objetivo:** Entrenar y evaluar modelos que expliquen el precio de Equipo 1 y Equipo 2 a partir de las materias primas.

---

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
sys.path.append('../src')

from modeling import split_data, train_linear_model, evaluate_model, get_feature_importance

PROCESSED_PATH = Path('../data/processed')
print('Librerías y módulos cargados.')

Librerías y módulos cargados.


## 1. Carga de Datos Procesados



In [2]:
df = pd.read_csv(PROCESSED_PATH / "historico_equipos_limpio.csv")

df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

print("Dataset procesado cargado correctamente")
print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]}")
print(f"Periodo: {df['Date'].min().date()} → {df['Date'].max().date()}")

df.head()

Dataset procesado cargado correctamente
Filas: 3,530
Columnas: 6
Periodo: 2010-01-04 → 2023-08-31


,Date,Price_X,Price_Y,Price_Z,Price_Equipo1,Price_Equipo2
0,2010-01-04,80.12,527.5,2225.25,434.73,931.73
1,2010-01-05,80.59,527.5,2246.50,449.97,968.56
2,2010-01-06,81.89,527.5,2302.50,444.48,960.51
3,2010-01-07,81.51,527.5,2306.50,440.90,960.14
4,2010-01-08,81.37,552.5,2261.25,448.82,949.55


## 2. Preparación de Variables (X, y)

In [3]:
# Variables predictoras
FEATURES = [
    "Price_X",
    "Price_Y",
    "Price_Z"
]

# Objetivos
TARGET_EQUIPO1 = "Price_Equipo1"
TARGET_EQUIPO2 = "Price_Equipo2"

print("Variables predictoras:")
print(FEATURES)

print("\nVariable objetivo Equipo 1:")
print(TARGET_EQUIPO1)

print("\nVariable objetivo Equipo 2:")
print(TARGET_EQUIPO2)

Variables predictoras:
['Price_X', 'Price_Y', 'Price_Z']

Variable objetivo Equipo 1:
Price_Equipo1

Variable objetivo Equipo 2:
Price_Equipo2


## 3. Entrenamiento de Modelos — Equipo 1

In [4]:
from modeling import (
    time_series_split_data,
    train_linear_model,
    evaluate_model
)

# División temporal
X_train, X_test, y_train, y_test, train_df, test_df = (
    time_series_split_data(
        df=df,
        feature_cols=FEATURES,
        target_col=TARGET_EQUIPO1,
        test_size=0.2
    )
)

print("Entrenamiento:", len(X_train))
print("Prueba:", len(X_test))

# Modelo lineal
modelo_equipo1 = train_linear_model(
    X_train,
    y_train,
    model_type="linear"
)

metricas_equipo1 = evaluate_model(
    modelo_equipo1,
    X_test,
    y_test
)

print("\nMétricas Equipo 1")
print(metricas_equipo1)

Entrenamiento: 2824
Prueba: 706

Métricas Equipo 1
{'RMSE': np.float64(10.3088), 'MAE': 8.8699, 'R2': 0.9913}


In [5]:
coeficientes = pd.DataFrame({
    "Variable": FEATURES,
    "Coeficiente": modelo_equipo1.coef_
})

coeficientes = coeficientes.sort_values(
    "Coeficiente",
    ascending=False
)

coeficientes

,Variable,Coeficiente
1,Price_Y,0.796755
0,Price_X,0.203392
2,Price_Z,0.001188


## 4. Entrenamiento de Modelos — Equipo 2

In [6]:
# División temporal
X_train, X_test, y_train, y_test, train_df, test_df = (
    time_series_split_data(
        df=df,
        feature_cols=FEATURES,
        target_col=TARGET_EQUIPO2,
        test_size=0.2
    )
)

# Modelo lineal
modelo_equipo2 = train_linear_model(
    X_train,
    y_train,
    model_type="linear"
)

metricas_equipo2 = evaluate_model(
    modelo_equipo2,
    X_test,
    y_test
)

print("Métricas Equipo 2")
print(metricas_equipo2)

Métricas Equipo 2
{'RMSE': np.float64(19.3198), 'MAE': 16.5514, 'R2': 0.9853}


In [12]:
import importlib
import modeling

importlib.reload(modeling)

from modeling import train_gradient_boosting_model, get_feature_importance

print("Módulo modeling recargado correctamente.")

Módulo modeling recargado correctamente.


In [13]:
gb_equipo2 = train_gradient_boosting_model(
    X_train,
    y_train
)

metricas_gb_equipo2 = evaluate_model(
    gb_equipo2,
    X_test,
    y_test
)

print("Métricas Gradient Boosting - Equipo 2")
print(metricas_gb_equipo2)

Métricas Gradient Boosting - Equipo 2
{'RMSE': np.float64(96.995), 'MAE': 57.4077, 'R2': 0.6289}


In [14]:
importance_equipo2 = get_feature_importance(
    gb_equipo2,
    FEATURES
)

importance_equipo2

,feature,importance
2,Price_Z,0.800184
1,Price_Y,0.192098
0,Price_X,0.007718


In [9]:
import modeling

dir(modeling)

['Dict',
 'GradientBoostingRegressor',
 'Lasso',
 'LinearRegression',
 'Ridge',
 'Tuple',
 '__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__spec__',
 'cross_val_score',
 'evaluate_model',
 'get_feature_importance',
 'mean_absolute_error',
 'mean_squared_error',
 'np',
 'pd',
 'r2_score',
 'split_data',
 'time_series_split_data',
 'train_linear_model',
 'train_test_split']

In [15]:
from modeling import train_gradient_boosting_model, get_feature_importance

gb_equipo2 = train_gradient_boosting_model(
    X_train,
    y_train
)

metricas_gb_equipo2 = evaluate_model(
    gb_equipo2,
    X_test,
    y_test
)

print(metricas_gb_equipo2)

{'RMSE': np.float64(96.995), 'MAE': 57.4077, 'R2': 0.6289}


## 5. Comparación de Modelos

In [16]:
# =====================================================
# COMPARACIÓN DE MODELOS
# =====================================================
#
# Objetivo:
# Comparar el desempeño de los modelos evaluados para
# cada equipo.
#
# Se utilizan las métricas:
# - RMSE
# - MAE
# - R²
#
# El mejor modelo será utilizado posteriormente en la
# fase de forecasting.
# =====================================================

comparacion_modelos = pd.DataFrame([
    {
        "Equipo": "Equipo 1",
        "Modelo": "Regresión Lineal",
        "RMSE": 10.3088,
        "MAE": 8.8699,
        "R2": 0.9913
    },
    {
        "Equipo": "Equipo 2",
        "Modelo": "Regresión Lineal",
        "RMSE": 19.3198,
        "MAE": 16.5514,
        "R2": 0.9853
    },
    {
        "Equipo": "Equipo 2",
        "Modelo": "Gradient Boosting",
        "RMSE": 96.9950,
        "MAE": 57.4077,
        "R2": 0.6289
    }
])

comparacion_modelos
'''### Hallazgos

- La regresión lineal obtuvo el mejor desempeño para ambos equipos.
- El modelo explica más del 98% de la variabilidad observada en los precios históricos.
- Gradient Boosting presentó un desempeño significativamente inferior para el Equipo 2.
- Esto sugiere que la relación entre materias primas y precios de los equipos es predominantemente lineal.

### Modelo seleccionado

Se selecciona la Regresión Lineal como modelo principal debido a:

- Mayor precisión predictiva.
- Mayor interpretabilidad.
- Menor complejidad.
- Mejor capacidad de explicación para el negocio.'''

,Equipo,Modelo,RMSE,MAE,R2
0,Equipo 1,Regresión Lineal,10.3088,8.8699,0.9913
1,Equipo 2,Regresión Lineal,19.3198,16.5514,0.9853
2,Equipo 2,Gradient Boosting,96.9950,57.4077,0.6289


## 6. Importancia de Variables

# 6. Importancia de Variables

## Objetivo

Identificar cuáles materias primas presentan mayor influencia sobre el comportamiento de cada equipo.

Esta etapa responde directamente a una de las preguntas principales del caso de negocio:

> ¿Qué materias primas explican los cambios en el costo de adquisición de los equipos?

In [ ]:
# =====================================================
# IMPORTANCIA DE VARIABLES - EQUIPO 1
# =====================================================

coeficientes

### Interpretación Equipo 1

Los coeficientes del modelo lineal muestran que:

- Price_Y es la variable dominante.
- Price_X tiene una influencia secundaria.
- Price_Z tiene un efecto prácticamente nulo.

Este resultado coincide con la matriz de correlación obtenida durante el análisis exploratorio.

In [ ]:
# =====================================================
# IMPORTANCIA DE VARIABLES - EQUIPO 2
# =====================================================

importance_equipo2

### Interpretación Equipo 2

El análisis mediante Gradient Boosting muestra que:

- Price_Z aporta aproximadamente el 80% de la capacidad explicativa.
- Price_Y aporta cerca del 19%.
- Price_X tiene una participación marginal.

Por lo tanto, Price_Z se identifica como la principal materia prima asociada al comportamiento del Equipo 2.

## 7. Conclusiones del Modelado

## Conclusiones principales

### Equipo 1

El análisis exploratorio y los modelos predictivos muestran una relación altamente consistente entre Price_Y y el costo del Equipo 1.

Resultados relevantes:

- Correlación de Pearson: 0.997
- R² del modelo lineal: 0.9913
- Variable más influyente: Price_Y

Se concluye que Price_Y es el principal determinante del comportamiento histórico del Equipo 1.

---

### Equipo 2

El análisis muestra una fuerte dependencia entre Price_Z y el costo del Equipo 2.

Resultados relevantes:

- Correlación de Pearson: 0.983
- R² del modelo lineal: 0.9853
- Importancia en Gradient Boosting: 80%

Se concluye que Price_Z es el principal determinante del comportamiento histórico del Equipo 2.

---

### Conclusión general

Los resultados demuestran que las materias primas contienen suficiente información para explicar y anticipar los costos de adquisición de los equipos.

La regresión lineal presentó el mejor desempeño para ambos casos, logrando explicar más del 98% de la variabilidad observada.

Por esta razón, se considera una metodología adecuada para soportar procesos de planeación financiera y estimación de costos futuros dentro del proyecto de construcción.